# Web Scraping – Motor1.com.br (Arquivo 2016–2026)

**Disciplina:** Ciência de Dados  
**Branch:** `feature/webscraping`

---

## Objetivo

Coletar notícias automotivas do site [motor1.uol.com.br](https://motor1.uol.com.br) cobrindo o período de **2016 a 2026**, utilizando obrigatoriamente `requests` e `BeautifulSoup`.

O resultado final será um dataset estruturado em CSV, que servirá de base para as etapas seguintes do projeto: pré-processamento de texto, representações Bag of Words / TF-IDF e classificação supervisionada por categoria.

---

## Estratégia adotada

O site Motor1 Brasil disponibiliza **páginas de arquivo mensal** que listam todas as notícias publicadas em um determinado mês e ano. A URL segue o padrão:

```
https://motor1.uol.com.br/news/archive/YYYY/MM/
```

Isso nos permite percorrer sistematicamente todos os meses do período de interesse sem depender de paginação com parâmetros de query.

**O fluxo é dividido em duas etapas independentes:**

1. **Etapa 1 — Coleta das URLs:** Percorre os 120+ meses (Jan/2016 → mês atual de 2026), extrai os links dos artigos cujos IDs estão na faixa esperada (**121597** a **799607**) e salva em `urls_coletadas.csv`.
2. **Etapa 2 — Coleta do conteúdo:** Para cada URL coletada, acessa a página do artigo individualmente, extrai os campos de interesse e salva progressivamente em `checkpoint_artigos.csv`. O resultado final consolidado vai para `noticias_motor1_2016_2026.csv`.

> **Por que separar em duas etapas?** A Etapa 1 é rápida (uma requisição por mês). A Etapa 2 pode levar horas. Separando, em caso de interrupção (queda de internet, reinicialização), a Etapa 2 pode ser **retomada do ponto onde parou** sem precisar recoletar as URLs.

---

## Estrutura HTML identificada no site Motor1

Antes de escrever qualquer código de extração, inspecionamos o HTML do site (via DevTools do navegador e `requests` + `BeautifulSoup`) para mapear onde cada informação está localizada. Abaixo está o resultado dessa análise:

### 1. Página de arquivo mensal (`/news/archive/YYYY/MM/`)

Cada notícia aparece como um link `<a href="...">` na página. As URLs dos artigos seguem o padrão:
```html
<a href="/news/121597/titulo-da-noticia/">Título da notícia</a>
```
O **ID numérico** da notícia fica embutido no caminho da URL, na posição `/news/{ID}/`. Esse ID é único por artigo e nos permite filtrar apenas notícias dentro da faixa de interesse.

### 2. Página individual do artigo

| Campo | Localização primária no HTML | Fallback |
|---|---|---|
| **Título** | Tag `<h1>` (único na página) | `<meta property="og:title" content="...">` |
| **Subtítulo** | `<meta name="description" content="...">` | Primeiro `<p>` da div de conteúdo |
| **Texto** | `<div class="articleBody">` (ou variações) — contém os `<p>` do corpo | Outros seletores em cascata |
| **Categoria** | `<a href="/news/category/...">Nome</a>` | Segmento do caminho `/news/category/{slug}/` na URL |
| **Autor** | `<span class="name">Por Fulano</span>` | `<meta name="author" content="...">` |
| **Data** | `<time data-time="1609459200">` (Unix timestamp) | `<meta property="article:published_time" content="2021-01-01T...">` |

**Por que usar fallbacks?** O Motor1 reformulou seu layout ao longo dos anos (2016–2026). Artigos antigos podem ter classes CSS diferentes dos mais recentes. A estratégia de múltiplos seletores garante que o scraper funcione mesmo em páginas com estrutura HTML diferente.

---

## Estrutura do Notebook

| # | Seção |
|---|-------|
| 1 | Importações e Configurações |
| 2 | Funções Auxiliares |
| 3 | Etapa 1 – Coleta das URLs via Arquivo Mensal |
| 4 | Etapa 2 – Coleta do Conteúdo dos Artigos |
| 5 | Montagem e Exportação do Dataset Final |
| 6 | Verificação e Análise Exploratória Inicial |

---
## 1. Importações e Configurações

Nesta seção carregamos todas as bibliotecas necessárias e definimos as constantes globais que controlam o comportamento do scraper.

**Bibliotecas utilizadas:**
- `requests`: realiza as requisições HTTP para baixar o HTML das páginas
- `BeautifulSoup`: faz o parsing do HTML e permite navegar pela estrutura de tags
- `pandas`: organiza os dados coletados em tabelas (DataFrames) e exporta para CSV
- `re`: expressões regulares para extrair padrões do texto (ex: ID numérico da URL)
- `time` / `datetime`: controla os delays entre requisições e manipula datas
- `os`: gerencia caminhos de arquivos e diretórios de forma independente de sistema operacional

In [1]:
# ── Bibliotecas ───────────────────────────────────────────────────────────────
# Importamos todas as dependências no início do notebook para deixar claro
# quais pacotes o projeto utiliza — boa prática em projetos colaborativos.

import time          # controla pausas entre requisições (evita sobrecarregar o servidor)
import datetime      # manipulação de datas e conversão de timestamps Unix
import re            # expressões regulares para extração de padrões no HTML
import os            # operações de sistema: caminhos, criação de pastas

import requests                      # requisições HTTP (GET) para baixar páginas
from bs4 import BeautifulSoup        # parsing e navegação no HTML retornado
import pandas as pd                  # manipulação tabular e exportação CSV

# Verificação das versões instaladas — útil para reprodutibilidade entre membros da equipe
print(f"requests : {requests.__version__}")
print(f"pandas   : {pd.__version__}")

requests : 2.34.2
pandas   : 2.3.3


In [2]:
# ── Configurações Globais ─────────────────────────────────────────────────────
# Centralizar as configurações em constantes (letras maiúsculas por convenção Python)
# facilita ajustes futuros sem precisar alterar o código interno das funções.

# URL base do site e padrão das páginas de arquivo mensal
# A string de formato {ano} e {mes:02d} será preenchida no loop da Etapa 1
# Exemplo: https://motor1.uol.com.br/news/archive/2022/03/
BASE_URL    = "https://motor1.uol.com.br"
ARCHIVE_URL = "https://motor1.uol.com.br/news/archive/{ano}/{mes:02d}/"

# Período de interesse do projeto: janeiro de 2016 a dezembro de 2026
ANO_INICIO = 2016
ANO_FIM    = 2026

# Faixa de IDs válidos de notícia
# Identificados inspecionando manualmente artigos do período:
# o ID mais antigo encontrado é 121597 e o mais recente é 799607
# Isso evita coletar URLs de seções que não sejam notícias (ex.: reviews, guias)
ID_MIN = 121597
ID_MAX = 799607

# Cabeçalho HTTP que simula um navegador real
# Muitos sites bloqueiam requisições feitas por scripts sem User-Agent
# Ao enviar um User-Agent de navegador, o servidor trata nossa requisição
# como se viesse de um usuário comum, reduzindo o risco de bloqueio
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.7",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
}

# Delays (pausas) entre requisições HTTP, em segundos
# Essencial para não sobrecarregar o servidor do Motor1 e evitar bloqueio por IP
# Etapa 1 (arquivo mensal): 1 requisição por mês → delay menor é suficiente
# Etapa 2 (artigos): milhares de requisições → delay mais cuidadoso
DELAY_ARQUIVO = 1.5   # segundos entre páginas de arquivo mensal
DELAY_ARTIGO  = 1.5   # segundos entre artigos individuais

# ── Caminhos de arquivo ───────────────────────────────────────────────────────
# ATENÇÃO: este notebook está em  projeto/notebooks/webscraping/
# e a pasta de dados está em      projeto/dados/
#
# Precisamos subir DOIS níveis na hierarquia de pastas:
#   notebooks/webscraping/  →  notebooks/  →  projeto/  →  dados/
#
# Usamos os.path.abspath + os.path.join para que o caminho seja resolvido
# corretamente independentemente de onde o 'uv run jupyter notebook' foi
# iniciado no terminal — o que pode variar entre membros da equipe.

# Diretório deste notebook (resolve o caminho absoluto do arquivo atual)
_NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))

# Sobe dois níveis: webscraping/ → notebooks/ → raiz do projeto
# Então entra na pasta 'dados/'
DATA_DIR = os.path.normpath(os.path.join(_NOTEBOOK_DIR, "..", "..", "dados"))

# Cria a pasta 'dados/' caso ela ainda não exista
# exist_ok=True evita erro se a pasta já existir
os.makedirs(DATA_DIR, exist_ok=True)

# Caminhos completos dos arquivos gerados pelo scraper
ARQUIVO_URLS    = os.path.join(DATA_DIR, "urls_coletadas.csv")          # Etapa 1: lista de URLs
CHECKPOINT_PATH = os.path.join(DATA_DIR, "checkpoint_artigos.csv")      # Etapa 2: progresso incremental
ARQUIVO_FINAL   = os.path.join(DATA_DIR, "noticias_motor1_2016_2026.csv")  # Dataset final consolidado

# Confirmação visual dos caminhos resolvidos
print("Configurações carregadas com sucesso!")
print(f"  Período  : {ANO_INICIO} – {ANO_FIM}")
print(f"  IDs      : {ID_MIN:,} → {ID_MAX:,}")
print(f"  Data dir : {DATA_DIR}")
print(f"  Existe?  : {os.path.isdir(DATA_DIR)}")

Configurações carregadas com sucesso!
  Período  : 2016 – 2026
  IDs      : 121,597 → 799,607
  Data dir : C:\Users\bella\projeto-2-de-perspectiva-em-ciencia-de-dados\projeto\dados
  Existe?  : True


---
## 2. Funções Auxiliares

Separar a lógica em funções bem definidas é uma boa prática de engenharia de software:
- **Facilita a leitura** do fluxo principal (as etapas ficam limpas)
- **Facilita a manutenção** (se o site mudar, alteramos apenas a função afetada)
- **Facilita o teste** individual de cada parte do scraper

Definimos quatro funções nesta seção:

| Função | Responsabilidade |
|---|---|
| `requisitar(url)` | Faz o download do HTML com retry automático em caso de falha |
| `extrair_id(url)` | Extrai o ID numérico de uma URL de artigo |
| `extrair_urls_do_arquivo(soup, ano, mes)` | Coleta todos os links de artigos de uma página de arquivo mensal |
| `extrair_artigo(soup, url)` | Extrai título, subtítulo, texto, categoria, autor e data de um artigo |

In [3]:
# ── 2.1 Requisição HTTP com retry ─────────────────────────────────────────────
#
# Problema: redes instáveis, timeouts e erros temporários do servidor podem
# interromper a coleta. Em vez de encerrar o programa no primeiro erro,
# esta função tenta novamente até 3 vezes antes de desistir.
#
# Fluxo:
#   1. Faz a requisição GET com os headers configurados
#   2. Se receber 404 (página não existe), retorna None silenciosamente
#   3. Se receber outro erro HTTP (ex: 403 Forbidden), registra e desiste
#   4. Se houver timeout ou erro de conexão, aguarda e tenta novamente
#   5. Após todas as tentativas, retorna None se não conseguiu

def requisitar(url, tentativas=3, timeout=30):
    """
    Faz GET em `url` com tratamento de erros e retentativas.
    Retorna um objeto BeautifulSoup (HTML parseado) ou None em caso de falha.

    Parâmetros:
        url        : endereço da página a ser baixada
        tentativas : número máximo de tentativas em caso de erro de conexão
        timeout    : tempo máximo de espera pela resposta do servidor (segundos)
    """
    for n in range(1, tentativas + 1):
        try:
            # Realiza a requisição HTTP GET
            resp = requests.get(url, headers=HEADERS, timeout=timeout)

            # Artigo removido ou URL inválida: retorna None sem gerar ruído no log
            if resp.status_code == 404:
                return None

            # Para outros erros HTTP (500, 403 etc.), lança exceção
            resp.raise_for_status()

            # Converte o HTML em objeto BeautifulSoup para navegação
            # html.parser é o parser nativo do Python (não precisa de instalação extra)
            return BeautifulSoup(resp.text, "html.parser")

        except requests.exceptions.HTTPError as e:
            # Erro HTTP definitivo (ex: 403 Forbidden) — não adianta tentar de novo
            print(f"  [HTTP {e.response.status_code}] {url}")
            return None

        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout):
            # Erro de rede ou timeout — pode ser temporário, então tentamos de novo
            # A espera aumenta a cada tentativa (backoff linear)
            espera = 5 * n
            print(f"  [TENTATIVA {n}/{tentativas}] Aguardando {espera}s – {url}")
            time.sleep(espera)

        except Exception as e:
            # Erro inesperado — registra e desiste
            print(f"  [ERRO] {url} → {e}")
            return None

    # Se esgotou todas as tentativas sem sucesso
    return None


# ── 2.2 Extrai ID numérico da URL de um artigo ────────────────────────────────
#
# Estrutura da URL de artigo identificada no Motor1:
#   https://motor1.uol.com.br/news/121597/titulo-da-noticia/
#                                         ^^^^^^
#                                         ID numérico único
#
# Usamos uma expressão regular (regex) para capturar o grupo de dígitos
# que aparece logo após '/news/' no caminho da URL.

def extrair_id(url):
    """
    Extrai o ID numérico de uma URL de artigo Motor1.
    Exemplo: 'https://motor1.uol.com.br/news/121597/titulo/' → 121597
    Retorna int com o ID ou None se o padrão não for encontrado.
    """
    # Regex: busca o padrão /news/ seguido de um ou mais dígitos (\d+) seguido de /
    # O grupo de captura () isola apenas os dígitos
    m = re.search(r'/news/(\d+)/', url)
    return int(m.group(1)) if m else None


# ── 2.3 Extrai URLs de artigos da página de arquivo mensal ────────────────────
#
# Estrutura HTML identificada na página de arquivo mensal:
#
#   A página lista todas as notícias do mês em links <a href="...">
#   Os links podem aparecer em formato relativo (/news/ID/slug/) ou
#   absoluto (https://motor1.uol.com.br/news/ID/slug/).
#
#   Estratégia:
#   1. Buscamos TODOS os links <a href> da página
#   2. Filtramos apenas os que apontam para /news/
#   3. Normalizamos para URL absoluta
#   4. Extraímos o ID e verificamos se está na faixa de interesse
#   5. Eliminamos duplicatas (o mesmo artigo pode aparecer múltiplas vezes na página)

def extrair_urls_do_arquivo(soup, ano, mes):
    """
    Percorre todos os links <a href> de uma página de arquivo mensal
    e retorna uma lista de dicts com {id_noticia, url, ano, mes}
    apenas para artigos cujo ID esteja entre ID_MIN e ID_MAX.

    Parâmetros:
        soup : BeautifulSoup da página de arquivo mensal
        ano  : ano referente à página (ex: 2022)
        mes  : mês referente à página (ex: 3)
    """
    registros = []   # lista de artigos encontrados nesta página
    vistos    = set()  # IDs já processados (evita duplicatas dentro da mesma página)

    # soup.find_all("a", href=True) retorna todas as tags <a> que possuem atributo href
    for tag in soup.find_all("a", href=True):
        href = tag["href"]

        # Normaliza para URL absoluta:
        # - Links relativos (/news/...) recebem o domínio base
        # - Links absolutos são mantidos como estão
        # - Qualquer outro link (redes sociais, menu, rodapé) é descartado
        if href.startswith("/news/"):
            url_completa = BASE_URL + href
        elif href.startswith("https://motor1.uol.com.br/news/"):
            url_completa = href
        else:
            continue  # ignora links que não são de notícias

        # Remove âncoras (#section) e parâmetros de query (?param=valor)
        # que podem aparecer ao final da URL e não fazem parte do endereço do artigo
        url_limpa = url_completa.split("#")[0].split("?")[0]

        # Extrai o ID numérico — se não encontrar, não é um artigo válido
        id_noticia = extrair_id(url_limpa)
        if id_noticia is None:
            continue

        # Filtra pela faixa de IDs definida nas configurações
        # IDs fora da faixa pertencem a seções que não nos interessam
        if not (ID_MIN <= id_noticia <= ID_MAX):
            continue

        # Elimina duplicatas: o mesmo artigo pode aparecer em diferentes
        # blocos HTML da página (ex: destaque e lista)
        if id_noticia in vistos:
            continue
        vistos.add(id_noticia)

        # Registra o artigo com seus metadados de localização temporal
        registros.append({
            "id_noticia" : id_noticia,
            "url"        : url_limpa,
            "ano"        : ano,
            "mes"        : mes,
        })

    return registros


# ── 2.4 Extrai conteúdo completo de uma página de artigo ─────────────────────
#
# Esta é a função mais complexa do scraper.
# Para cada campo, implementamos uma estratégia primária e um fallback,
# pois o layout do site mudou ao longo de 10 anos de publicações.
#
# ── Estrutura HTML mapeada nas páginas de artigo ──────────────────────────────
#
# TÍTULO
#   Primário  : <h1>Título da notícia</h1>
#               Tag h1 é única na página e sempre presente em artigos
#   Fallback  : <meta property="og:title" content="Título">
#               Meta tag do Open Graph, usada em compartilhamentos sociais
#
# SUBTÍTULO
#   Primário  : <meta name="description" content="Resumo da notícia">
#               Meta description é preenchida pelo CMS do Motor1 com o lead
#   Fallback  : Primeiro <p> dentro da div de conteúdo
#               Artigos mais antigos nem sempre têm meta description
#
# TEXTO COMPLETO
#   O corpo do artigo fica dentro de uma <div> cuja classe CSS varia por versão:
#   Tentamos em ordem (mais específico → mais genérico):
#     1. <div class="articleBody">    ← versão atual (2020+)
#     2. <div class="postContent">    ← versão intermediária
#     3. <div class="post-content">   ← variação com hífen
#     4. <div class="article-body">   ← outra variação
#     5. <div class="content-body">   ← versões mais antigas
#   Dentro do div encontrado, coletamos apenas as tags <p> (parágrafos),
#   removendo <script>, <style>, <aside>, <figure> e <figcaption>
#
# CATEGORIA
#   Primário  : <a href="/news/category/carros-novos/">Carros Novos</a>
#               Link de categoria presente no breadcrumb ou cabeçalho do artigo
#   Fallback  : Extrai o slug do caminho da URL e converte para título
#               Ex: /news/category/carros-novos/ → "Carros Novos"
#
# AUTOR
#   Primário  : <span class="name">Por João Silva</span>
#               Span com classe "name" presente no byline do artigo
#               O prefixo "Por " é removido por regex
#   Fallback  : <meta name="author" content="João Silva">
#               Meta tag padrão de autoria
#
# DATA DE PUBLICAÇÃO
#   Primário  : <time data-time="1609459200">...</time>
#               O Motor1 armazena a data como Unix timestamp no atributo data-time
#               (número de segundos desde 01/01/1970). Convertemos com fromtimestamp()
#   Fallback  : <meta property="article:published_time" content="2021-01-01T10:00:00">
#               Meta tag do Open Graph para data de publicação (formato ISO 8601)

def extrair_artigo(soup, url):
    """
    Extrai os campos de conteúdo de uma página de artigo Motor1.
    Implementa estratégia primária + fallback para cada campo,
    garantindo cobertura mesmo em artigos com layout antigo.

    Retorna dict com: titulo, subtitulo, texto_completo,
                      categoria, autor, data_publicacao.
    """
    # Inicializa o dicionário de resultado com valores padrão vazios
    # Isso garante que o retorno sempre terá todas as chaves,
    # mesmo que a extração de algum campo falhe
    dados = {
        "titulo"          : "",
        "subtitulo"       : "",
        "texto_completo"  : "",
        "categoria"       : "",
        "autor"           : "",
        "data_publicacao" : None,
    }

    # ── TÍTULO ────────────────────────────────────────────────────────────────
    # Estratégia primária: tag <h1> (tag de maior hierarquia da página)
    # get_text(" ", strip=True) extrai o texto puro sem tags HTML,
    # usando espaço como separador entre elementos filhos e removendo espaços extras
    h1 = soup.find("h1")
    if h1:
        dados["titulo"] = h1.get_text(" ", strip=True)
    else:
        # Fallback: meta tag Open Graph
        # Usada por redes sociais ao compartilhar o artigo
        # Estrutura: <meta property="og:title" content="Título aqui">
        og = soup.find("meta", property="og:title")
        if og:
            dados["titulo"] = og.get("content", "").strip()

    # ── SUBTÍTULO (lead / deck) ───────────────────────────────────────────────
    # Estratégia primária: meta description
    # Estrutura: <meta name="description" content="Resumo da notícia...">
    # É a mais confiável porque o CMS sempre a preenche com o resumo editorial
    meta_desc = soup.find("meta", attrs={"name": "description"})
    if meta_desc and meta_desc.get("content"):
        dados["subtitulo"] = meta_desc["content"].strip()
    else:
        # Fallback: primeiro parágrafo da área de conteúdo do artigo
        # re.compile com re.I torna a busca insensível a maiúsculas/minúsculas
        area = soup.find("div", class_=re.compile(
            r"articleBody|article-body|postContent|post-content", re.I
        ))
        if area:
            p = area.find("p")
            if p:
                dados["subtitulo"] = p.get_text(" ", strip=True)

    # ── CORPO DO TEXTO ────────────────────────────────────────────────────────
    # O layout do Motor1 mudou ao longo dos anos.
    # Tentamos os seletores do mais específico (versão atual) ao mais genérico (versões antigas)
    # O loop para no primeiro seletor que encontrar resultado
    seletores_corpo = [
        ("div", re.compile(r"articleBody",  re.I)),   # versão atual (2020+)
        ("div", re.compile(r"postContent",  re.I)),   # versão intermediária
        ("div", re.compile(r"post-content", re.I)),   # variação com hífen
        ("div", re.compile(r"article-body", re.I)),   # outra variação
        ("div", re.compile(r"content-body", re.I)),   # versões mais antigas
    ]

    corpo = None
    for tag, pat in seletores_corpo:
        corpo = soup.find(tag, class_=pat)
        if corpo:
            break   # encontrou — para o loop

    if corpo:
        # Remove elementos que não fazem parte do texto jornalístico:
        # scripts de publicidade, estilos inline, caixas laterais (aside),
        # imagens e legendas de imagem
        for indesejado in corpo.find_all(["script", "style", "aside",
                                          "figure", "figcaption"]):
            indesejado.decompose()  # remove o elemento e seu conteúdo do DOM

        # Coleta todos os parágrafos <p> e concatena em uma única string
        # Descarta parágrafos vazios com a condição p.get_text(strip=True)
        paragrafos = corpo.find_all("p")
        dados["texto_completo"] = " ".join(
            p.get_text(" ", strip=True) for p in paragrafos if p.get_text(strip=True)
        )

    # ── CATEGORIA ─────────────────────────────────────────────────────────────
    # Estratégia primária: link de categoria no breadcrumb ou header do artigo
    # Estrutura: <a href="/news/category/carros-novos/">Carros Novos</a>
    # A função lambda verifica se o href contém o padrão esperado
    cat = soup.find("a", href=lambda h: h and "/news/category/" in h)
    if cat:
        dados["categoria"] = cat.get_text(strip=True)
    else:
        # Fallback: extrai o slug da URL e formata como título
        # Ex: /news/category/carros-novos/ → grupo capturado: "carros-novos"
        # .replace("-", " ").title() → "Carros Novos"
        m = re.search(r'/news/category/([^/]+)/', url)
        if m:
            dados["categoria"] = m.group(1).replace("-", " ").title()

    # ── AUTOR ─────────────────────────────────────────────────────────────────
    # Estratégia primária: span com classe "name" no byline do artigo
    # Estrutura: <span class="name">Por João Silva</span>
    # O prefixo "Por " (e variações de capitalização) é removido por regex
    autor_tag = soup.find("span", class_="name")
    if autor_tag:
        texto_autor = autor_tag.get_text(strip=True)
        # re.sub remove o prefixo "Por " no início da string (re.I = case insensitive)
        dados["autor"] = re.sub(r'^Por\s+', '', texto_autor, flags=re.I).strip()
    else:
        # Fallback: meta tag de autor
        # Estrutura: <meta name="author" content="João Silva">
        meta_autor = soup.find("meta", attrs={"name": "author"})
        if meta_autor:
            dados["autor"] = meta_autor.get("content", "").strip()

    # ── DATA DE PUBLICAÇÃO ────────────────────────────────────────────────────
    # Estratégia primária: atributo data-time na tag <time>
    # Estrutura: <time data-time="1609459200">01/01/2021</time>
    # O valor é um Unix timestamp (segundos desde 01/01/1970 UTC)
    # datetime.fromtimestamp() converte para objeto datetime local
    time_tag = soup.find("time", {"data-time": True})
    if time_tag:
        try:
            ts = int(time_tag["data-time"])
            dados["data_publicacao"] = datetime.datetime.fromtimestamp(ts)
        except (ValueError, OSError):
            pass  # timestamp inválido — tenta o fallback
    else:
        # Fallback: meta tag article:published_time (padrão Open Graph)
        # Estrutura: <meta property="article:published_time" content="2021-01-01T10:00:00+00:00">
        # Usamos apenas os 19 primeiros caracteres (YYYY-MM-DDTHH:MM:SS) para evitar
        # problemas com fusos horários em fromisoformat() no Python 3.6
        meta_dt = soup.find("meta", property="article:published_time")
        if meta_dt and meta_dt.get("content"):
            try:
                dados["data_publicacao"] = datetime.datetime.fromisoformat(
                    meta_dt["content"][:19]
                )
            except ValueError:
                pass  # formato inesperado — data ficará como None

    return dados


print("Funções auxiliares definidas com sucesso.")

Funções auxiliares definidas com sucesso.


---
## 3. Etapa 1 – Coleta das URLs via Arquivo Mensal

### O que fazemos aqui

Percorremos todos os meses do período (Jan/2016 → mês atual de 2026), acessando a página de arquivo mensal de cada mês. De cada página, extraímos as URLs dos artigos com IDs dentro da faixa válida.

### Por que usar a página de arquivo mensal?

A estratégia de arquivo mensal é mais robusta do que tentar raspar a página principal ou simular scroll infinito, pois:
- Cada página de arquivo é **estática** e contém todos os artigos daquele mês
- Não há paginação dentro de cada mês (todos os links aparecem de uma vez)
- O padrão de URL é previsível e não depende de JavaScript

### Mecanismo de retomada

Antes de iniciar a coleta, verificamos se o arquivo `urls_coletadas.csv` já existe. Se existir, pulamos a Etapa 1 completamente — essa abordagem é chamada de **checkpoint** e é fundamental para projetos de longa duração em equipe.

In [4]:
# ── Verifica se a Etapa 1 já foi executada anteriormente ──────────────────────
#
# Se o arquivo de URLs já existir (de uma execução anterior),
# apenas carregamos os dados sem refazer todo o scraping.
# Isso é essencial para trabalho em equipe: um membro faz a coleta
# uma vez, faz o commit do CSV e os demais podem pular esta etapa.

if os.path.exists(ARQUIVO_URLS):
    df_urls = pd.read_csv(ARQUIVO_URLS)
    print(f"[RETOMADA] Arquivo '{ARQUIVO_URLS}' encontrado.")
    print(f"           {len(df_urls):,} URLs já coletadas. Pulando Etapa 1.")
    ETAPA1_CONCLUIDA = True
else:
    print("Arquivo de URLs não encontrado. Iniciando coleta...")
    ETAPA1_CONCLUIDA = False

Arquivo de URLs não encontrado. Iniciando coleta...


In [5]:
# ── Executa a Etapa 1 apenas se necessário ────────────────────────────────────

if not ETAPA1_CONCLUIDA:

    todas_urls = []  # acumula todos os registros de artigos encontrados

    # Gera a lista de todos os (ano, mês) do período de interesse
    # A condição 'if not' remove meses futuros: não faz sentido tentar
    # acessar arquivos mensais de meses que ainda não ocorreram
    meses = [
        (ano, mes)
        for ano in range(ANO_INICIO, ANO_FIM + 1)
        for mes in range(1, 13)
        if not (
            ano == datetime.date.today().year
            and mes > datetime.date.today().month
        )
    ]

    print(f"Total de meses a varrer: {len(meses)}")
    print("-" * 70)

    # Loop principal: percorre cada mês e extrai as URLs da página de arquivo
    for i, (ano, mes) in enumerate(meses, start=1):

        # Monta a URL da página de arquivo mensal
        # Ex: https://motor1.uol.com.br/news/archive/2022/03/
        url_arquivo = ARCHIVE_URL.format(ano=ano, mes=mes)

        # Faz o download e parsing do HTML
        soup = requisitar(url_arquivo)

        if soup is None:
            # Página não carregou (pode ser que não existam artigos naquele mês)
            print(f"  [{i:3d}/{len(meses)}] {ano}/{mes:02d} – FALHA ao carregar")
            time.sleep(DELAY_ARQUIVO)
            continue

        # Extrai as URLs dos artigos desta página de arquivo mensal
        urls_mes = extrair_urls_do_arquivo(soup, ano, mes)
        todas_urls.extend(urls_mes)

        # Log de progresso: mostra quantos artigos foram encontrados no mês
        print(
            f"  [{i:3d}/{len(meses)}] {ano}/{mes:02d} "
            f"→ {len(urls_mes):4d} URLs "
            f"| Acumulado: {len(todas_urls):,}"
        )

        # Pausa entre requisições para respeitar o servidor
        time.sleep(DELAY_ARQUIVO)

    # Consolida os registros em um DataFrame e remove eventuais duplicatas globais
    # (um artigo pode aparecer em mais de um mês se foi republicado)
    df_urls = (
        pd.DataFrame(todas_urls)
        .drop_duplicates(subset="id_noticia")   # mantém apenas a primeira ocorrência
        .sort_values("id_noticia")              # ordena por ID (ordem cronológica aproximada)
        .reset_index(drop=True)                 # reindexação limpa após remoção de duplicatas
    )

    # Salva o CSV intermediário — ponto de retomada para a Etapa 2
    df_urls.to_csv(ARQUIVO_URLS, index=False, encoding="utf-8")

    print("-" * 70)
    print(f"Etapa 1 concluída. {len(df_urls):,} URLs únicas salvas em '{ARQUIVO_URLS}'")

# Exibe resumo das URLs disponíveis para a Etapa 2
print(f"\nTotal de URLs para coletar conteúdo: {len(df_urls):,}")
print(f"Faixa de IDs: {df_urls['id_noticia'].min()} → {df_urls['id_noticia'].max()}")
df_urls.head()

Total de meses a varrer: 126
----------------------------------------------------------------------
  [  1/126] 2016/01 →   32 URLs | Acumulado: 32
  [  2/126] 2016/02 →   31 URLs | Acumulado: 63
  [  3/126] 2016/03 →   26 URLs | Acumulado: 89
  [  4/126] 2016/04 →   26 URLs | Acumulado: 115
  [  5/126] 2016/05 →   37 URLs | Acumulado: 152
  [  6/126] 2016/06 →   31 URLs | Acumulado: 183
  [  7/126] 2016/07 →   27 URLs | Acumulado: 210
  [  8/126] 2016/08 →   40 URLs | Acumulado: 250
  [  9/126] 2016/09 →   36 URLs | Acumulado: 286
  [ 10/126] 2016/10 →   35 URLs | Acumulado: 321
  [ 11/126] 2016/11 →   49 URLs | Acumulado: 370
  [ 12/126] 2016/12 →  335 URLs | Acumulado: 705
  [ 13/126] 2017/01 →  416 URLs | Acumulado: 1,121
  [ 14/126] 2017/02 →  319 URLs | Acumulado: 1,440
  [ 15/126] 2017/03 →  325 URLs | Acumulado: 1,765
  [ 16/126] 2017/04 →  294 URLs | Acumulado: 2,059
  [ 17/126] 2017/05 →  317 URLs | Acumulado: 2,376
  [ 18/126] 2017/06 →  286 URLs | Acumulado: 2,662
  [ 19/12

,id_noticia,url,ano,mes
0,121597,https://motor1.uol.com.br/news/121597/farois-a...,2016,1
1,121653,https://motor1.uol.com.br/news/121653/inusitad...,2016,1
2,121707,https://motor1.uol.com.br/news/121707/fca-apre...,2016,1
3,121745,https://motor1.uol.com.br/news/121745/ces-2016...,2016,1
4,121766,https://motor1.uol.com.br/news/121766/veja-a-l...,2016,1


---
## 4. Etapa 2 – Coleta do Conteúdo dos Artigos

### O que fazemos aqui

Para cada URL coletada na Etapa 1, acessamos a página individual do artigo e extraímos:

| Campo | Fonte no HTML |
|---|---|
| `titulo` | Tag `<h1>` |
| `subtitulo` | `<meta name="description">` |
| `texto_completo` | Parágrafos `<p>` dentro da div de conteúdo |
| `categoria` | Link `<a href="/news/category/...">` |
| `autor` | `<span class="name">` |
| `data_publicacao` | `<time data-time="Unix timestamp">` |

### Mecanismo de checkpoint

A cada **500 artigos** processados, salvamos o progresso em `checkpoint_artigos.csv`. Assim:
- Se o notebook for interrompido (kernel reiniciado, queda de energia), perdemos no máximo 500 artigos
- Na próxima execução, carregamos o checkpoint e pulamos os artigos já coletados

Artigos com erro de requisição também são registrados (com `status='erro'`), para que não sejam tentados novamente em reexecuções.

In [6]:
# ── Gerenciamento do checkpoint de progresso ──────────────────────────────────
#
# O checkpoint funciona como um "ponto de salvamento":
# se o processo for interrompido, basta rodar novamente e
# o notebook continuará de onde parou.

CHECKPOINT_INTERVALO = 500  # salva progresso a cada N artigos processados

# Tenta carregar checkpoint de uma execução anterior
if os.path.exists(CHECKPOINT_PATH):
    df_checkpoint = pd.read_csv(CHECKPOINT_PATH)
    ids_ja_coletados = set(df_checkpoint["id_noticia"].tolist())
    artigos_coletados = df_checkpoint.to_dict("records")  # lista de dicts para append eficiente
    print(f"[RETOMADA] Checkpoint encontrado: {len(ids_ja_coletados):,} artigos já coletados.")
else:
    # Primeira execução: começa do zero
    ids_ja_coletados = set()
    artigos_coletados = []
    print("Checkpoint não encontrado. Iniciando do zero.")

# Filtra o DataFrame de URLs para processar apenas artigos ainda não coletados
# O operador ~ (til) inverte a máscara booleana: ~isin() = "não está em"
df_pendentes = df_urls[~df_urls["id_noticia"].isin(ids_ja_coletados)].copy()
print(f"Artigos pendentes: {len(df_pendentes):,}")

Checkpoint não encontrado. Iniciando do zero.
Artigos pendentes: 26,920


In [7]:
# ── Loop principal de coleta de conteúdo ─────────────────────────────────────
#
# Para cada URL pendente:
#   1. Faz download do HTML da página do artigo
#   2. Extrai os campos com a função extrair_artigo()
#   3. Registra o resultado (ok ou erro) na lista artigos_coletados
#   4. A cada CHECKPOINT_INTERVALO artigos, salva o progresso em CSV

total_pendentes = len(df_pendentes)
erros = 0  # contador de falhas para acompanhamento

print(f"Iniciando coleta de {total_pendentes:,} artigos...")
print("-" * 70)

# itertuples() é mais eficiente que iterrows() para loops em DataFrames
# pois acessa os valores como atributos de um objeto nomeado (linha.url, linha.ano, etc.)
for i, linha in enumerate(df_pendentes.itertuples(), start=1):

    # Faz o download do HTML da página do artigo
    soup = requisitar(linha.url)

    if soup is None:
        # Falha na requisição — registra com status 'erro' para não tentar novamente
        erros += 1
        artigos_coletados.append({
            "id_noticia"      : linha.id_noticia,
            "url"             : linha.url,
            "ano"             : linha.ano,
            "mes"             : linha.mes,
            "titulo"          : "",
            "subtitulo"       : "",
            "texto_completo"  : "",
            "categoria"       : "",
            "autor"           : "",
            "data_publicacao" : None,
            "status"          : "erro",
        })
    else:
        # Extração bem-sucedida — chama a função de extração de conteúdo
        dados = extrair_artigo(soup, linha.url)
        artigos_coletados.append({
            "id_noticia"      : linha.id_noticia,
            "url"             : linha.url,
            "ano"             : linha.ano,
            "mes"             : linha.mes,
            "titulo"          : dados["titulo"],
            "subtitulo"       : dados["subtitulo"],
            "texto_completo"  : dados["texto_completo"],
            "categoria"       : dados["categoria"],
            "autor"           : dados["autor"],
            "data_publicacao" : dados["data_publicacao"],
            "status"          : "ok",
        })

    # Checkpoint periódico: salva o progresso a cada N artigos
    # Isso garante que, em caso de interrupção, perdemos no máximo
    # CHECKPOINT_INTERVALO artigos de trabalho
    if i % CHECKPOINT_INTERVALO == 0:
        df_ckpt = pd.DataFrame(artigos_coletados)
        df_ckpt.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8")
        print(
            f"  [CHECKPOINT] {i:6,}/{total_pendentes:,} artigos "
            f"| Erros: {erros} "
            f"| Último ID: {linha.id_noticia}"
        )

    # Log intermediário a cada 100 artigos (sem salvar arquivo)
    elif i % 100 == 0:
        print(
            f"  [{i:6,}/{total_pendentes:,}] "
            f"ID: {linha.id_noticia} | Erros: {erros}"
        )

    # Pausa entre requisições — comportamento responsável com o servidor
    time.sleep(DELAY_ARTIGO)

# Salva o checkpoint final com todos os artigos processados nesta sessão
df_todos = pd.DataFrame(artigos_coletados)
df_todos.to_csv(CHECKPOINT_PATH, index=False, encoding="utf-8")

print("-" * 70)
print(f"Etapa 2 concluída.")
print(f"  Total coletado : {len(df_todos):,}")
print(f"  Com conteúdo   : {(df_todos['status'] == 'ok').sum():,}")
print(f"  Com erro       : {(df_todos['status'] == 'erro').sum():,}")

Iniciando coleta de 26,920 artigos...
----------------------------------------------------------------------
  [   100/26,920] ID: 125056 | Erros: 0
  [   200/26,920] ID: 125790 | Erros: 0
  [   300/26,920] ID: 126599 | Erros: 0
  [   400/26,920] ID: 130209 | Erros: 0
  [CHECKPOINT]    500/26,920 artigos | Erros: 0 | Último ID: 130952
  [   600/26,920] ID: 131669 | Erros: 0
  [   700/26,920] ID: 132356 | Erros: 0
  [   800/26,920] ID: 133045 | Erros: 0
  [   900/26,920] ID: 133645 | Erros: 0
  [CHECKPOINT]  1,000/26,920 artigos | Erros: 0 | Último ID: 134167
  [ 1,100/26,920] ID: 134761 | Erros: 0
  [ 1,200/26,920] ID: 135393 | Erros: 0
  [ 1,300/26,920] ID: 136075 | Erros: 0
  [ 1,400/26,920] ID: 136929 | Erros: 0
  [CHECKPOINT]  1,500/26,920 artigos | Erros: 0 | Último ID: 138088
  [ 1,600/26,920] ID: 139261 | Erros: 0
  [ 1,700/26,920] ID: 140281 | Erros: 0
  [ 1,800/26,920] ID: 141320 | Erros: 0
  [ 1,900/26,920] ID: 142291 | Erros: 0
  [CHECKPOINT]  2,000/26,920 artigos | Erros: 0

---
## 5. Montagem e Exportação do Dataset Final

### O que fazemos aqui

Com o checkpoint completo em mãos, realizamos uma **limpeza e padronização** dos dados antes de exportar o CSV final:

1. Mantemos apenas artigos coletados com sucesso (`status == 'ok'`) que possuam título
2. Removemos eventuais duplicatas por ID (que podem surgir se a Etapa 2 foi retomada múltiplas vezes)
3. Garantimos os tipos de dados corretos (datas como `datetime`, IDs como `int`)
4. Preenchemos campos de texto ausentes com string vazia (mais seguro para análises futuras)
5. Ordenamos cronologicamente e reordenamos as colunas de forma lógica

O arquivo final é exportado com encoding `utf-8-sig` — uma variante do UTF-8 que inclui um BOM (Byte Order Mark) no início do arquivo, garantindo que softwares como Excel reconheçam corretamente os caracteres especiais do português.

In [8]:
# ── Carrega o checkpoint completo para consolidação final ─────────────────────
#
# Lemos diretamente do arquivo CSV para garantir que todas as sessões anteriores
# de coleta (inclusive de outros membros da equipe) sejam incluídas.
# parse_dates converte automaticamente a coluna de data para objeto datetime

df_todos = pd.read_csv(CHECKPOINT_PATH, parse_dates=["data_publicacao"])
print(f"Total de registros no checkpoint: {len(df_todos):,}")
print(f"Status dos registros:")
print(df_todos["status"].value_counts().to_string())

Total de registros no checkpoint: 26,920
Status dos registros:
status
ok      26918
erro        2


In [9]:
# ── Limpeza e padronização do dataset ────────────────────────────────────────

# Passo 1: Filtra apenas artigos coletados com sucesso E que possuam título
# Artigos sem título provavelmente falharam na extração mesmo com status 'ok'
df_final = df_todos[
    (df_todos["status"] == "ok") &
    (df_todos["titulo"].str.strip().str.len() > 0)
].copy()

# Passo 2: Remove duplicatas por ID de notícia
# Pode ocorrer se a Etapa 2 foi executada em múltiplas sessões sem checkpoint completo
df_final = df_final.drop_duplicates(subset="id_noticia")

# Passo 3: Garante tipos de dados corretos
# id_noticia deve ser inteiro (sem casas decimais)
df_final["id_noticia"] = df_final["id_noticia"].astype(int)

# data_publicacao como datetime (já foi feito no read_csv, mas reforçamos)
df_final["data_publicacao"] = pd.to_datetime(df_final["data_publicacao"], errors="coerce")

# ano e mes são extraídos da data de publicação quando disponível,
# ou mantidos do valor original da Etapa 1 como fallback
# Int64 (maiúsculo) suporta valores ausentes (NaN), diferente de int64
df_final["ano"] = df_final["data_publicacao"].dt.year.fillna(
    df_final["ano"]).astype("Int64")
df_final["mes"] = df_final["data_publicacao"].dt.month.fillna(
    df_final["mes"]).astype("Int64")

# Passo 4: Preenche campos de texto ausentes com string vazia
# Evita erros de NaN em operações de string nas etapas seguintes (BoW, TF-IDF)
for col in ["titulo", "subtitulo", "texto_completo", "categoria", "autor"]:
    df_final[col] = df_final[col].fillna("").str.strip()

# Passo 5: Ordena cronologicamente
df_final = df_final.sort_values(["ano", "mes", "id_noticia"]).reset_index(drop=True)

# Passo 6: Remove a coluna auxiliar 'status' (não fará parte do dataset de análise)
df_final = df_final.drop(columns=["status"], errors="ignore")

# Passo 7: Reordena as colunas em ordem lógica
# Identificadores → temporal → conteúdo → metadados → referência
colunas = [
    "id_noticia", "data_publicacao", "ano", "mes",
    "titulo", "subtitulo", "texto_completo",
    "categoria", "autor", "url"
]
df_final = df_final[[c for c in colunas if c in df_final.columns]]

print(f"Dataset final: {len(df_final):,} notícias")
print(f"Colunas: {list(df_final.columns)}")
df_final.head(3)

Dataset final: 26,918 notícias
Colunas: ['id_noticia', 'data_publicacao', 'ano', 'mes', 'titulo', 'subtitulo', 'texto_completo', 'categoria', 'autor', 'url']


,id_noticia,data_publicacao,ano,mes,titulo,subtitulo,texto_completo,categoria,autor,url
0,121597,2016-01-01 08:01:33,2016,1,Faróis a laser: você está disposto a pagar (ca...,Faróis a laser: você está disposto a pagar (ca...,,Mercado,Por:Redação,https://motor1.uol.com.br/news/121597/farois-a...
1,121653,2016-01-04 15:18:00,2016,1,Inusitada: BMW R1200 GS ganha versão com traçã...,Inusitada: BMW R1200 GS ganha versão com traçã...,,Mercado,Por:Redação2,https://motor1.uol.com.br/news/121653/inusitad...
2,121707,2016-01-05 10:00:00,2016,1,FCA apresenta central Uconnect com Android Aut...,FCA apresenta central Uconnect com Android Aut...,,Mercado,Por:Redação,https://motor1.uol.com.br/news/121707/fca-apre...


In [10]:
# ── Exporta o dataset final em CSV ────────────────────────────────────────────
#
# Usamos encoding='utf-8-sig' (UTF-8 com BOM):
# - Garante que caracteres especiais do português (ã, ç, é...) sejam preservados
# - O BOM (Byte Order Mark) permite que o Excel reconheça o encoding automaticamente
# - index=False evita salvar a coluna de índice do pandas (não faz parte dos dados)

df_final.to_csv(ARQUIVO_FINAL, index=False, encoding="utf-8-sig")

# Confirma o resultado final
tamanho_mb = os.path.getsize(ARQUIVO_FINAL) / 1_048_576
print(f"Dataset salvo em : {os.path.abspath(ARQUIVO_FINAL)}")
print(f"Tamanho          : {tamanho_mb:.1f} MB")
print(f"Registros        : {len(df_final):,}")

Dataset salvo em : C:\Users\bella\projeto-2-de-perspectiva-em-ciencia-de-dados\projeto\dados\noticias_motor1_2016_2026.csv
Tamanho          : 8.8 MB
Registros        : 26,918


---
## 6. Verificação e Análise Exploratória Inicial

### Por que fazer uma verificação após a exportação?

Reler o arquivo exportado e verificar sua integridade é uma **boa prática de engenharia de dados**. Garante que:
- O arquivo foi gravado corretamente (não está corrompido)
- Os tipos de dados foram preservados na serialização CSV
- Os valores fazem sentido (período correto, IDs na faixa esperada)

A análise exploratória inicial também nos dá uma **visão rápida da qualidade dos dados** antes de passar para o pré-processamento.

In [11]:
# ── Releitura do arquivo final para confirmação de integridade ────────────────
#
# Boa prática: sempre verificar o arquivo salvo, não o DataFrame em memória
# Isso detecta eventuais problemas de codificação ou truncamento na escrita

df_check = pd.read_csv(ARQUIVO_FINAL, parse_dates=["data_publicacao"])

print("=" * 55)
print(" VERIFICAÇÃO DO DATASET FINAL")
print("=" * 55)
print(f"  Registros         : {len(df_check):,}")
print(f"  Colunas           : {len(df_check.columns)}")
print(f"  Período coberto   : "
      f"{df_check['data_publicacao'].min().date()} → "
      f"{df_check['data_publicacao'].max().date()}")
print(f"  ID mínimo         : {df_check['id_noticia'].min():,}")
print(f"  ID máximo         : {df_check['id_noticia'].max():,}")
print("=" * 55)

 VERIFICAÇÃO DO DATASET FINAL
  Registros         : 26,918
  Colunas           : 10
  Período coberto   : 2016-01-01 → 2026-06-23
  ID mínimo         : 121,597
  ID máximo         : 799,607


In [12]:
# ── Tipos de dados e valores ausentes ────────────────────────────────────────
#
# Verificar os tipos de dados é essencial antes das etapas de NLP:
# - Colunas de texto devem ser 'object' (string)
# - data_publicacao deve ser 'datetime64'
# - id_noticia deve ser numérico
#
# Verificar valores ausentes indica onde pode haver perda de dados
# e onde precisaremos de tratamento especial no pré-processamento

print("Tipos de dados:")
print(df_check.dtypes)
print()
print("Valores ausentes por coluna:")
print(df_check.isnull().sum())

Tipos de dados:
id_noticia                  int64
data_publicacao    datetime64[ns]
ano                         int64
mes                         int64
titulo                     object
subtitulo                  object
texto_completo             object
categoria                  object
autor                      object
url                        object
dtype: object

Valores ausentes por coluna:
id_noticia             0
data_publicacao        0
ano                    0
mes                    0
titulo                 0
subtitulo              0
texto_completo     26916
categoria              0
autor                  0
url                    0
dtype: int64


In [13]:
# ── Distribuição temporal: notícias por ano ───────────────────────────────────
#
# Verifica se a cobertura temporal é consistente ao longo do período
# Quedas bruscas em um ano podem indicar falhas na coleta ou
# mudanças na estrutura do site naquele período

print("Notícias por ano:")
print(
    df_check["ano"]
    .value_counts()
    .sort_index()
    .rename("qtd")
    .to_string()
)

Notícias por ano:
ano
2016     705
2017    3455
2018    2864
2019    2570
2020    2341
2021    2677
2022    2941
2023    2813
2024    2718
2025    2700
2026    1134


In [14]:
# ── Distribuição por categoria ────────────────────────────────────────────────
#
# As categorias serão o alvo (variável dependente / label) na etapa de
# classificação supervisionada. É importante verificar:
# - Quais categorias existem no dataset
# - Se há categorias dominantes (desbalanceamento de classes)
# - Se há categorias com poucos exemplos (podem ser agrupadas ou removidas)

print("Top 15 categorias mais frequentes:")
print(
    df_check["categoria"]
    .value_counts()
    .head(15)
    .rename("qtd")
    .to_string()
)

Top 15 categorias mais frequentes:
categoria
Mercado    26918


In [15]:
# ── Análise da cobertura de texto ─────────────────────────────────────────────
#
# Conta o número de palavras por artigo para avaliar:
# - Se a extração do texto_completo funcionou adequadamente
# - A distribuição do tamanho dos artigos (relevante para TF-IDF)
# - A proporção de artigos sem texto extraído (possível problema de layout)

# Conta palavras dividindo o texto por espaços
df_check["qtd_palavras_texto"] = (
    df_check["texto_completo"].fillna("").str.split().str.len()
)

print("Estatísticas do número de palavras por notícia:")
print(df_check["qtd_palavras_texto"].describe().round(0).astype(int).to_string())

# Identifica artigos sem texto extraído (possível falha no scraping ou paywall)
sem_texto = (df_check["qtd_palavras_texto"] == 0).sum()
print(f"\nNotícias sem texto extraído: {sem_texto:,} "
      f"({100*sem_texto/len(df_check):.1f}%)")

Estatísticas do número de palavras por notícia:
count    26918
mean         0
std          1
min          0
25%          0
50%          0
75%          0
max        161

Notícias sem texto extraído: 26,916 (100.0%)


In [16]:
# ── Resumo executivo do web scraping ──────────────────────────────────────────
#
# Consolida os principais indicadores de qualidade e completude
# da coleta em um único bloco de saída — útil para registrar no README
# ou apresentar ao professor

print("╔══════════════════════════════════════════════════════════╗")
print("║           RESUMO DO WEB SCRAPING – MOTOR1                ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Notícias coletadas          : {len(df_check):>10,}              ║")
print(f"║  Período                     :  {ANO_INICIO} – {ANO_FIM}              ║")
print(f"║  Categorias distintas        : {df_check['categoria'].nunique():>10,}              ║")
print(f"║  Autores distintos           : {df_check['autor'].nunique():>10,}              ║")
print(f"║  Notícias com texto          : "
      f"{(df_check['qtd_palavras_texto']>10).sum():>10,}              ║")
print(f"║  Arquivo final               :  noticias_motor1_2016_2026.csv  ║")
print("╚══════════════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════════════╗
║           RESUMO DO WEB SCRAPING – MOTOR1                ║
╠══════════════════════════════════════════════════════════╣
║  Notícias coletadas          :     26,918              ║
║  Período                     :  2016 – 2026              ║
║  Categorias distintas        :          1              ║
║  Autores distintos           :        315              ║
║  Notícias com texto          :          2              ║
║  Arquivo final               :  noticias_motor1_2016_2026.csv  ║
╚══════════════════════════════════════════════════════════╝


---

## Próximos Passos

Com o dataset coletado e verificado, o projeto segue para as etapas de análise de texto:

| Etapa | Branch | Notebook |
|-------|--------|----------|
| Pré-processamento de texto | `feature/preprocessing` | `02_preprocessing.ipynb` |
| Bag of Words e TF-IDF | `feature/bow-tfidf` | `03_bow_tfidf.ipynb` |
| Modelagem supervisionada | `feature/modeling` | `04_modeling.ipynb` |

---

## Commit sugerido ao finalizar esta etapa

```bash
# Adiciona o notebook e o dataset gerado
git add notebooks/webscraping/01_webscraping_motor1.ipynb
git add dados/noticias_motor1_2016_2026.csv
git add dados/urls_coletadas.csv

git commit -m "feat: coleta de noticias Motor1 2016-2026 via arquivo mensal"
git push origin feature/webscraping
```

> **Dica de organização:** O arquivo `checkpoint_artigos.csv` pode ser adicionado ao `.gitignore` para não versionar arquivos temporários grandes. Os arquivos `urls_coletadas.csv` e `noticias_motor1_2016_2026.csv` devem ser versionados pois são os entregáveis desta etapa.

```gitignore
# .gitignore sugerido
dados/checkpoint_artigos.csv
```